In [17]:
pd.read_csv('Solimo - Aug 26',encoding='latin')

ParserError: Error tokenizing data. C error: Expected 1 fields in line 44, saw 2


In [8]:
#Classification
import pandas as pd
import numpy as np
import tkinter as tk
from tkinter import filedialog,messagebox
from datetime import datetime
import os
import datetime
from tqdm import tqdm
#def var(x):



now = datetime.datetime.now().date()
var=pd.read_csv('Mee Mee - Sep23.txt',sep='\t',encoding='latin')## update the path
var.sort_values(by='attribute_value_list',inplace=True)
var.rename(columns={'ï»¿asin':'asin'},inplace=True)
#var['attribute_value_list'].replace('','',regex=True,inplace=True)
var2=var[var['comp_flag']==1].drop_duplicates()
var2['group']=(var2['ptd']+var2['brand_name']+var2['attribute_value_list']).rank(method='dense')
var2[['attribute_value_list','group']]
var2['flag']=var2.groupby('group')['asin'].transform("count")
var2['parent_asin1']=var2['parent_asin']
var2['count_parent']=var2.groupby('group')['parent_asin'].transform("nunique")
var2['recommended_parent_asin']=np.nan
var2['recommended_parent_asin']=np.where((var2['flag']==1) & (var2['parent_asin'].notna()),"Make Standalone","")

var2['recommended_parent_asin']=np.where((var2['flag']>1) & (var2['count_parent']==0),"Create New Parent",var2['recommended_parent_asin'])
print(1)
var3=var2[var2['flag']>1]
var3.fillna(0,inplace=True)
#

var3['parent_asin1']=var3['parent_asin']
var31=var3[var3['count_parent']==1]
var32=var31[['parent_asin','group']].drop_duplicates()
var32['ra']=var32.groupby('parent_asin').cumcount()+1
var31.drop(columns='recommended_parent_asin',inplace=True)
var31=pd.merge(var31,var32[(var32['ra']==1) & (var32['parent_asin']!=0)][['parent_asin','group']].rename(columns={'parent_asin':'recommended_parent_asin'}),how='left',on='group')

var31['recommended_parent_asin']=np.where((var31['recommended_parent_asin'].isna()) | (var31['recommended_parent_asin']=="") ,"Create New Parent",var31['recommended_parent_asin'])
var11=var3[(var3['count_parent']>1)]
var11.loc[var11['parent_asin'].isin(set(var31['recommended_parent_asin'])),'parent_asin']=0
result_list = []
skip_set = set()

for group_name, x in tqdm(var11.groupby("group")):
    x = x.copy()
    x.loc[x['parent_asin'].isin(skip_set),'parent_asin']=np.nan

    if x['count_parent'].max() > 1:
        y = x[(x['parent_asin'].notna()) & (x['parent_asin']!=0)]
        if not y.empty:
            best_asin = y.loc[y['gvs'].idxmax(), 'parent_asin']
            x['recommended_parent_asin'] = best_asin
            skip_set.add(best_asin)
        else:
            x['recommended_parent_asin'] = "Create New Parent"
    else:
        y = x[(x['parent_asin'].notna()) & (x['parent_asin']!=0)]
        if not y.empty:
            best_asin = y.loc[y['gvs'].idxmax(), 'parent_asin']
            x['recommended_parent_asin'] = best_asin
            skip_set.add(best_asin)
        else:
            x['recommended_parent_asin'] = "Create New Parent"

    result_list.append(x)

# # Combine results
var11 = pd.concat(result_list, ignore_index=True)
var4=pd.concat([var31,var11,var2[var2['flag']==1],var3[var3['count_parent']==0]])
var4['recommended_parent_asin']=np.where(var4['recommended_parent_asin'].isna(),"Create New Parent",var4['recommended_parent_asin'])
var4['Status']=np.where((var4['recommended_parent_asin']=='Create New Parent') | (var4['recommended_parent_asin']=='Make Standalone') | (var4['recommended_parent_asin'].isna()),\
                                                                                                     var4['recommended_parent_asin'],\
                           np.where((var4['recommended_parent_asin']==var4['parent_asin']),'To be Retained','Add to Existing'))
var4['Status']=np.where((var4['count_parent']==0) & (var4['recommended_parent_asin']=="") & (var4['flag']==1),'No Action',var4['Status'])
var4.drop_duplicates(inplace=True)
var4.drop(columns='parent_asin',inplace=True)
var4.rename(columns={'parent_asin1':'parent_asin'},inplace=True)
var4['parent_asin'].replace(0,np.nan,inplace=True)
var4.drop(columns='flag',inplace=True)
var4
var4['group_cou']=var4.groupby('group',as_index=False)['asin'].transform('count')
var4['asin_cou']=var4.groupby(['group','asin'],as_index=False)['asin'].transform('count')
var4['new_stat']=''
var4['new_stat']=np.where((var4['group_cou']>var4['asin_cou']) & (var4['asin_cou']>=2) & (var4['Status']=='Add to Existing'),'yes',np.where((var4['group_cou']==var4['asin_cou']) & (var4['Status']=='Add to Existing') & (var4['Status']!='No Action'),'make_standalone',var4['new_stat']))
var4.drop(columns=['group_cou','asin_cou'],inplace=True)
var4.to_csv(str(now)+'-Raw-Output.csv',index=False)
print('Done')

/tmp/ipykernel_29361/4119160504.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  var3.fillna(0,inplace=True)
/tmp/ipykernel_29361/4119160504.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  var3['parent_asin1']=var3['parent_asin']
/tmp/ipykernel_29361/4119160504.py:38: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  var31.drop(columns='recommended_parent_asin',inplace=True)

1


100%|██████████| 55/55 [00:00<00:00, 999.96it/s]

Done



/tmp/ipykernel_29361/4119160504.py:81: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  var4['parent_asin'].replace(0,np.nan,inplace=True)


In [24]:
var4['group_cou']=var4.groupby('group',as_index=False)['asin'].transform('count')
var4['asin_cou']=var4.groupby(['group','asin'],as_index=False)['asin'].transform('count')
var4['new_stat']=''
var4['new_stat']=np.where((var4['group_cou']>var4['asin_cou']) & (var4['asin_cou']>=2) & (var4['Status']=='Add to Existing'),'yes',np.where((var4['group_cou']==var4['asin_cou']) & (var4['Status']!='Make Standalone') & (var4['Status']!='No Action'),'make_standalone',var4['new_stat']))
var4.drop(columns=['group_cou','asin_cou'],inplace=True)
var4.to_csv(str(now)+'-Raw-Output.csv',index=False)

In [25]:
#var5[var5['asin']=='B00N3KYAEG']
var4[~var4['Status'].isin(['Make Standalone','No Action','To be Retained','Add to Existing'])]
var4[var4['asin']=='B0786BVJVF']


,asin,ptd,brand_name,attribute_list,attribute_value_list,attribute_list_var,attribute_value_list_var,item_name,comp_flag,orphan_variation,...,run_dt,gl_product_group_desc,recommendation,parent_rep,group,parent_asin,count_parent,recommended_parent_asin,Status,new_stat
1219,B0786BVJVF,BASSINET,Mee Mee,brand_name|furniture_finish|material_type,mee mee|wooden|wood,color_name|material_type_varattr,large|wood,Mee Mee Spacious Foldable Baby Wooden Cot with...,1,Y,...,9/7/2025 0:00,gl_baby_product,standalone_no_action,NaN,140.0,NaN,0,,No Action,


In [3]:
import pandas as pd
import numpy as np
import tqdm
from typing import *

import numpy as np
import pandas as pd
from display import *
from call_bedrock_fast_claude_v3_py import *
import sagemaker
import boto3
from sagemaker import get_execution_role

system='''Consider yourself as a Catalog Category expert.Your task is to read the list of titles in <inp></inp> provided in prompt and identify attribute which can be used to create variations from given list of attribute provided in <att></att>\
in prompt.Each title is separated by "<>" in \
<inp></inp> represents individual title and attribute is separated by "+". Pick titles in <inp></inp> and use your intelligence and own valid value repository to identify for the attributes using which they can be varieted  \
from list of attributes in <att></att> in prompt.\
only if different valid value for one attribute is found then\
respond attribute in <res></res>.Incase you get different valid value for more than one attribute then respond in comma separated format.If you get valid values for an attribute and it is same for all1 group then do consider that attribute for variation theme.\
If you dont find any attribte then respond NA.\
Make sure to output your answer for one input in <res></res>.Stop responding after output of attribute.\
Please do not give any output apart from list provided in <att></att>. You can refer <example></example> for reference input out put instruction.You can refer <thinking></thinking> for approach.\

<example>
<att>color_name+material_type+size_name</att>
<inp>Cello Metallo Chair (Black)<>Cello. Metallo Plastic Cafeteria Chairs (Standard, Orange) - Set of 2</inp>
<thinking>As per instruction variation theme is color_name+material_type+size_name. In <inp></inp> I can see two titles. Checking my valid value repository I can see different value for for color_name, valid value is not present in all titles for material_type,\ 
valid value is not present in all titles for size_name. Hence I will output variation theme as 'color_name'.</thinking>
<res>color_name</res>
</example>

<example>
<att>color_name+size_name+team_name</att>
<inp>'Cello Sleek Stainless Steel Hot and Cold Double Walled Water Bottle (600ml, Red)<>Cello Sleek Stainless Steel Hot and Cold Double Walled Water Bottle (600ml, Silver)'</inp>
<thinking>Variation theme is color_name+size_name+team_name.In <inp></inp> I can see two titles. Checking my valid value repository I can see different value for for color_name i.e red,silver,valid value for size_name is same as 600ml,\
valid value is not present in all titles for team_name.As per instruction since non distinct valid values are present for only color_name hence correct output for variation theme will be 'color_name'.</thinking>

'''.strip()

prompt2='''<att>{a}</att>
<inp>{inp}</inp>
'''.strip()

var4['TRP']=np.where(~var4['attribute_list'].str.startswith('brand_name'),var4['attribute_list'].replace(['\|brand_name'],'',regex=True),var4['attribute_list'])
var4['TRP']=np.where(~var4['TRP'].str.contains('brand_name'),'brand_name|'+var4['TRP'],var4['TRP'])
var4['TRP'].replace('\|','+',regex=True,inplace=True)
var4['Variation_theme']=var4['attribute_list_var'].replace(['_varattr','\|'],['','+'],regex=True)


df1=var4[~var4['Status'].isin(['Make Standalone','No Action','To be Retained','Add to Existing'])]
# var_th=pd.read_csv('variation_theme.csv')
# var_th.rename(columns={'PTD':'ptd'},inplace=True)
# df1=pd.merge(df1,var_th,on='ptd',how='left')
# trp=pd.read_csv('TRP configuration for Entire POD.csv')[['PT','title_mandatory_headers_list_parent']]
# trp.rename(columns={'PT':'ptd','title_mandatory_headers_list_parent':'TRP'},inplace=True)
# trp=trp[trp['TRP'].notna()]
# df1=pd.merge(df1,trp,on='ptd',how='left')


#df1=df1[df1['group']==1698]

from bs4 import BeautifulSoup as soup
def ext(x,tag_name='p'):
    r=soup(str(x))
    c=r.find_all(tag_name)
    return str(c)

df2=df1.groupby(['group','Variation_theme','TRP'],as_index=False)['item_name'].apply(lambda x:'<>'.join(x))
df2['TRP'].replace('\|','+',regex=True,inplace=True)

df2['prompt1']=df2.apply(lambda x:prompt2.format(a=x['TRP'],inp=x['item_name']),axis=1).astype('str')
df2['prompt2']=df2.apply(lambda x:prompt2.format(a=x['Variation_theme'],inp=x['item_name']),axis=1).astype('str')
ad=pd.Series()


df2['Variation_Theme']=call_bedrock_fast(
            prompts=df2['prompt2'],  ## REPLACE THIS WITH YOUR PROMPT!!
            model="anthropic.claude-3-haiku-20240307-v1:0",
            system=system,  ## Only works with Claude v3
            max_new_tokens=200,
            temperature=1.0,
            #top_k=5 
        
        #top_p=1.0

            
        )


#b=soup.find_all('<res>')

df2['Variation_theme_Main_output']=df2['Variation_Theme'].apply(ext,tag_name='res')
df2['Variation_theme_Main_output'].replace(['\[<res>','</res>\]'],"",regex=True,inplace=True)



system='''Consider yourself as a Catalog Category expert.Your task is to read the list of titles in <inp></inp>\
and create a generalize title as per the TRP in <att></att>.Remove all other values from title which are not matching the TRP in <att><att>.\
.Each title is separated by "<>" in\
<inp></inp> represents individual title and attribute is separated by "+" . Pick titles in <inp></inp> and use your intelligence and own\
valid value repository to identify the attribute which needs to be retained\
from list of attributes in <att></att>.\
use your judgement similar to the following the <thinking></thinking> in last <example></example>. 
Respond the new title in <res></res>.\
If you dont find any title then respond NA.\
If you dont find any attribute value in any of the titles then respond "value_missing(attr)".If you find multiple vlaid values for same attribute in different title then resepond NA
Make sure to output your answer for one input in <res></res> only.Please make sure to not output in format of <res>val</res>,<res>val2</res>.You can refer <example></example> for reference input out put instruction.\

<example>
<att>item_type_name + compatible_phone_models + material_type</att>
<inp>Casotec Rangoli Paisley Art Design Printed Silicon Soft TPU Back Case Cover for 10.or D2</inp>
<res>Casotec Printed Silicon Soft TPU Back Case Cover for 10.or D2</res>
</example>

<example>
<att>item_type_name + compatible_phone_models + material_type</att>
<inp>Casotec Basic Case for 10.or D2 4G (Silicone_Multicolor)</inp>
<res> Casotec Basic Case for 10.or D2 4G</res>
</example>

<example>
<att>item_type_name + compatible_phone_models + material_type</att>
<inp>Casotec Bad Color Shape Design Printed Silicon Soft TPU Back Case Cover for Xolo Era<>
Casotec Basic Case for Xolo Era (Silicone_Multicolor)<>
Casotec Basic Case for Xolo Era (Silicone_Multicolor)
</inp>
<thinking>From the input given as per TRP I can find Back Case as most relevant for item_type_name, Xolo Era for compatible_devices
,Silicon Soft TPU Silicon Soft TPU  for material. Removing all extra data and creating best general title will be "Casotec Printed Silicon Soft TPU Back Case Cover for Xolo Era"</thinking>
<res>Casotec Printed Silicon Soft TPU Back Case Cover for Xolo Era</res>
</example>

<example>
<att>brand_name+material_type+model_name</att>
<inp>Bosch Hand Blender MS1BG1020I 400 W with Beaker and Chopper (Black)<>Bosch Hand Blender MS1WR0000I 300 W (White)</inp>
<thinking> From the input given as per TRP I can find brand as Bosch, no relevant values for material_type, no values for model_name. As instructed since attribute value is missing in title the correct output will be "value_missing(material_type,model_name)"
<res>value_missing(material_type,model_name)</res>
</example>

<example>
'<att>brand_name+material_type</att>\n<inp>Amazon Brand - Umi Brass Hanuman Idol, 8 X 5 X 4 Inches<>Amazon Brand - Umi Romantic Love Couple Showpiece with Lighting Gift for Love Couple Wife Girlfriend, Teddy Bear Showpiece, Anniversary Wedding Valentine Day Gift for Him Her (Pink)<>Amazon Brand - Umi Polyresin Pagdi Ganesha Murti Idol Statue for Car Dashboard, Ganesha Chaturthi Decoration Gift Item<>Amazon Brand - Umi Ganesh Idol for Car Dashboard, Ganesha Ganpati Idol for Home Gift, Terracotta Statue Murti for Car Home Decor Temple Puja Room Gift</inp>'
<thinking> From the input given as per TRP I can find brand as Amazon Brand - Umi, there are different relevant values in different title. As instructed since there are more then 1 relevane value for material type I recommend title as "NA"
<res>NA</res>
</example>

'''.strip()


prompt2='''<att>{a}</att>
<inp>{inp}</inp>
'''.strip()


import pandas as pd
#df1=pd.read_excel('Casotec - parent title generation.xlsx')
#sets=['Set'+str(i) for i in range(4227,len(df1))]
#print(sets)
#df1=df1[df1['Set'].isin(sets)]
#df1=df1[df1['PT']=='ELECTRONIC_DEVICE_SKIN']

# df4=df1[df1['Set']=='Set1']
# inp="<>".join(list(df4['Title']))
# a=df1['TRP'].unique()
# prompt=system.format(a=a,inp=inp)

#system
# df2=df1.groupby(['Set','TRP'],as_index=False)['Title'].apply(lambda x:'<>'.join(x))
# df2['TRP'].replace('\|','+',regex=True,inplace=True)

# df2['prompt']=df2.apply(lambda x:prompt2.format(a=x['TRP'],inp=x['Title']),axis=1).astype('str')
#df1['prompt']=df1.apply(lambda x:)
#df2


ad=pd.Series()

df2['Title']=call_bedrock_fast(
            prompts=df2['prompt1'],  ## REPLACE THIS WITH YOUR PROMPT!!
            model="anthropic.claude-3-haiku-20240307-v1:0",
            system=system,  ## Only works with Claude v3
            max_new_tokens=200,
            temperature=1.0,
        
        #top_p=1.0

            )

df2['Title_Main_output']=df2['Title'].apply(ext,tag_name='res')
df2['Title_Main_output'].replace(['\[<res>','</res>\]'],"",regex=True,inplace=True)
df1=pd.merge(df1,df2[['group','Title_Main_output','Variation_theme_Main_output']],on='group',how='left')
# df2.to_csv('Title_Casotec.csv',index=False)
df1.to_csv('Model_output.csv',index=False)
print('Done')
df2

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


/tmp/ipykernel_20945/1486790798.py:46: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  var4['TRP'].replace('\|','+',regex=True,inplace=True)
/tmp/ipykernel_20945/1486790798.py:69: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=Tru

Done


,group,Variation_theme,TRP,item_name,prompt1,prompt2,Variation_Theme,Variation_theme_Main_output,Title,Title_Main_output
0,28.0,color_name+size_name,brand_name+material_type+model_name,Amazon Brand - Solimo School Bag for Kids Boys...,<att>brand_name+material_type+model_name</att>...,<att>color_name+size_name</att>\n<inp>Amazon B...,"<res>color_name,size_name</res>","color_name,size_name",<res>Amazon Brand - Solimo School Bag for Kids...,Amazon Brand - Solimo School Bag for Kids Boys...
1,34.0,color_name+size_name,brand_name+material_type+model_name,Amazon Brand - Solimo fairy Unicorn School Bag...,<att>brand_name+material_type+model_name</att>...,<att>color_name+size_name</att>\n<inp>Amazon B...,"<res>color_name,size_name</res>","color_name,size_name",<res>Amazon Brand - Solimo Unicorn School Bag ...,Amazon Brand - Solimo Unicorn School Bag for G...
2,55.0,color_name+size_name,brand_name+material_type+model_name,Amazon Brand - Solimo Kids Velvet School Bag s...,<att>brand_name+material_type+model_name</att>...,<att>color_name+size_name</att>\n<inp>Amazon B...,"<res>color_name,size_name</res>","color_name,size_name",<res>Amazon Brand - Solimo Kids Velvet School ...,Amazon Brand - Solimo Kids Velvet School Bag s...
3,72.0,color_name+size_name,brand_name+material_type+model_name,Amazon Brand - Solimo Velvet Toddler Backpack ...,<att>brand_name+material_type+model_name</att>...,<att>color_name+size_name</att>\n<inp>Amazon B...,"<res>color_name,size_name</res>","color_name,size_name",<res>Amazon Brand - Solimo Velvet Toddler Back...,Amazon Brand - Solimo Velvet Toddler Backpack ...
4,98.0,color_name+size_name,brand_name+item_shape+material_type+room_type,Amazon Brand - Solimo 4 Piece Storage Basket S...,<att>brand_name+item_shape+material_type+room_...,<att>color_name+size_name</att>\n<inp>Amazon B...,"<res>color_name,size_name</res>","color_name,size_name",<res>Amazon Brand - Solimo Plastic Storage Bas...,"Amazon Brand - Solimo Plastic Storage Basket, ..."
...,...,...,...,...,...,...,...,...,...,...
7971,35432.0,color_name+frame_type+size_name,brand_name+frame_material_type+item_shape+room...,Amazon Brand - Solimo Pre-Laminated Pine Wood ...,<att>brand_name+frame_material_type+item_shape...,<att>color_name+frame_type+size_name</att>\n<i...,"<res>frame_type,size_name</res>","frame_type,size_name",<res>Amazon Brand - Solimo Pre-Laminated Pine ...,Amazon Brand - Solimo Pre-Laminated Pine Wood ...
7972,35435.0,color_name+frame_type+size_name,brand_name+frame_material_type+item_shape+room...,Amazon Brand - Solimo Koala Bear Painting with...,<att>brand_name+frame_material_type+item_shape...,<att>color_name+frame_type+size_name</att>\n<i...,"<res>color_name,frame_type,size_name</res>","color_name,frame_type,size_name",<res>Amazon Brand - Solimo Painting with Frame...,Amazon Brand - Solimo Painting with Frame
7973,35438.0,color_name+frame_type+size_name,brand_name+frame_material_type+item_shape+room...,Amazon Brand - Solimo Butterfly Painting with ...,<att>brand_name+frame_material_type+item_shape...,<att>color_name+frame_type+size_name</att>\n<i...,"<res>color_name,frame_type,size_name</res>","color_name,frame_type,size_name",<res>Amazon Brand - Solimo Canvas Painting wit...,Amazon Brand - Solimo Canvas Painting with Frame
7974,35453.0,color_name+frame_type+size_name,brand_name+frame_material_type+item_shape+room...,Amazon Solimo - Wooden wall dÃ©cor<>Amazon Bra...,<att>brand_name+frame_material_type+item_shape...,<att>color_name+frame_type+size_name</att>\n<i...,"<res>color_name,frame_type</res>","color_name,frame_type",<res>Amazon Solimo Wooden Wall Decor</res>,Amazon Solimo Wooden Wall Decor


In [36]:
# var4['attribute_list_var'].replace('_varattr','',inplace=True,regex=True)

var5=df1[['asin','attribute_list_var','attribute_value_list_var']]
var5['attribute_list_var'].replace('_varattr','',inplace=True,regex=True)
var5['attribute_list_var_split']=var5['attribute_list_var'].str.split('|')
var5['attribute_value_list_var_split']=var5['attribute_value_list_var'].str.split('|')
var5['attribute_value_list_var_split'].fillna(0,inplace=True)
var5=var5[var5['attribute_value_list_var']!=0]
def explode(x):

    keys=x['attribute_list_var_split']
    values=x['attribute_value_list_var_split']
    max_len=len(keys)
    result=[]
    for i in range(max_len):
        # print (x['asin'])
        # print(i)
        # #print(len(values))
        # print(values)
        val=values[i] if i<len((values)) else ''
        result.append(f'{keys[i]}&&{val}')
        print 
    return result
var5['attr_col']=var5.apply(explode,axis=1)
var6=var5[['asin','attr_col']].explode('attr_col',ignore_index=True)
var6[['attribute','value']]=var6['attr_col'].str.split('&&',expand=True).rename(columns={0:'attribute',1:'value'})
var6[var6['attribute']=='parent_asin']
mapping_file=pd.read_excel("Sample pre-filled template for Create new families.xlsx",sheet_name='Master mapping')[['attribute','attribute_name']]
var6=pd.merge(var6,mapping_file.drop_duplicates(),on='attribute',how='left')
var7=var6.pivot_table(index='asin',columns='attribute_name',values='value',aggfunc=lambda x: ''.join(x))
var7.insert(loc=0,column='asin',value=var7.index)
var7.reset_index(drop=True,inplace=True)
# for i in var7.columns:
#     if i.find('#1')>1:
#         var7.drop(columns=i,inplace=True)
df1=pd.merge(df1,var7,how='left',on='asin')
df1['recommended_parent_asin'].replace('Create New parent','Create New Parent',regex=True,inplace=True)
df1['recommended_parent_asin']=np.where(df1['recommended_parent_asin']=='Create New Parent',df1['brand_name']+df1['group'].astype('str'),df1['recommended_parent_asin'])
display(df1)
df1.to_csv('for_audit-'+str(now)+'.csv',index=False)

/tmp/ipykernel_20945/3916776926.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  var5['attribute_list_var'].replace('_varattr','',inplace=True,regex=True)
/tmp/ipykernel_20945/3916776926.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  var5['attribute_list_var'].replace('_varattr','',inplace=True,regex=True)
/tmp/ipykernel_20945/39167769

,asin,parent_asin,ptd,brand_name,attribute_list,attribute_value_list,attribute_list_var,attribute_value_list_var,item_name,comp_flag,...,color.value,frame#1.type#1.value_y,material#1.value_y,material.value,material_type_free#1.value_y,occasion_type#1.value_y,size#1.value_y,size.value,team_name#1.value_y,team_name.value_y
0,B098B51D9N,NaN,CELLULAR_PHONE_CASE,Amazon Brand - Solimo,brand_name|compatible_phone_models|form_factor...,amazon brand - solimo|10.or e|basic case|silic...,color_name_varattr,multicolor,Amazon Brand - Solimo Basic Case for 10.or E (...,1,...,multicolor,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,B098B7QD2S,NaN,CELLULAR_PHONE_CASE,Amazon Brand - Solimo,brand_name|compatible_phone_models|form_factor...,amazon brand - solimo|10.or e|basic case|silic...,color_name_varattr,multicolor,Amazon Brand - Solimo Basic Case for 10.or E (...,1,...,multicolor,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,B08W58Y7YG,NaN,CELLULAR_PHONE_CASE,Amazon Brand - Solimo,brand_name|compatible_phone_models|form_factor...,amazon brand - solimo|2020|basic case|plastic|...,color_name_varattr,multicolor,Amazon Brand - Solimo Designer Game Remote 3D ...,1,...,multicolor,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,B08SKDZ4B1,NaN,CELLULAR_PHONE_CASE,Amazon Brand - Solimo,brand_name|compatible_phone_models|form_factor...,amazon brand - solimo|2020|basic case|plastic|...,color_name_varattr,multicolor,Amazon Brand - Solimo Designer Golden Stars UV...,1,...,multicolor,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,B08W58HGSB,NaN,CELLULAR_PHONE_CASE,Amazon Brand - Solimo,brand_name|compatible_phone_models|form_factor...,amazon brand - solimo|2020|basic case|plastic|...,color_name_varattr,multicolor,Amazon Brand - Solimo Designer Peacock Feather...,1,...,multicolor,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7218,B0B34Q2HT3,NaN,CELLULAR_PHONE_CASE,Amazon Brand - Solimo,brand_name|compatible_phone_models|form_factor...,amazon brand - solimo|xiomi 12 pro 5g|basic ca...,color_name_varattr,multicolor,Amazon Brand- Solimo Basic Case for Xiomi 12 P...,1,...,multicolor,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7219,B0D1VFHQV9,B0DHRQ9WD8,STORAGE_BAG,Amazon Brand - Solimo,brand_name|closure_type|item_shape|material_ty...,amazon brand - solimo|zipper|rectangular|cotto...,color_name|size_name_varattr,cream|pack of 20,Amazon Brand - Solimo Saree Bags | Clothes Bag...,1,...,cream,NaN,NaN,NaN,NaN,pack of 20,pack of 20,pack of 20,NaN,NaN
7220,B0D1VFXNV8,NaN,STORAGE_BAG,Amazon Brand - Solimo,brand_name|closure_type|item_shape|material_ty...,amazon brand - solimo|zipper|rectangular|cotto...,color_name|size_name_varattr,cream|pack of 15,Amazon Brand - Solimo Hanging Saree Bag | Cott...,1,...,cream,NaN,NaN,NaN,NaN,pack of 15,pack of 15,pack of 15,NaN,NaN
7221,B0BV9F8F53,NaN,BOTTLE,Amazon Brand - Solimo,bottle_type|brand_name|material_type,wide-neck|amazon brand - solimo|polyethylene t...,color_name|size_name|team_name_varattr,multicolor|set of 3,Amazon Brand - Solimo Premium Plastic Water Bo...,1,...,multicolor,NaN,NaN,NaN,NaN,set of 3,set of 3,set of 3,,


In [37]:
df1[df1['Variation_theme'].str.contains('frame')]
#var6
df1[df1['asin']=='B0DGQ3D1S9']
var5[var5['asin']=='B0DGQ3D1S9']
var6[var6['asin']=='B0DGQ3D1S9'].pivot_table(index='asin',columns='attribute_name',values='value',aggfunc=lambda x: ''.join(x))
var7[var7['asin']=='B0DGQ3D1S9']
#df1[df1['asin']=='B0DGQ3D1S9']
var6[var6['asin']=='B0DGQ3D1S9']
df1[df1['asin']=='B098B51D9N']

,asin,parent_asin,ptd,brand_name,attribute_list,attribute_value_list,attribute_list_var,attribute_value_list_var,item_name,comp_flag,...,color.value,frame#1.type#1.value_y,material#1.value_y,material.value,material_type_free#1.value_y,occasion_type#1.value_y,size#1.value_y,size.value,team_name#1.value_y,team_name.value_y
0,B098B51D9N,NaN,CELLULAR_PHONE_CASE,Amazon Brand - Solimo,brand_name|compatible_phone_models|form_factor...,amazon brand - solimo|10.or e|basic case|silic...,color_name_varattr,multicolor,Amazon Brand - Solimo Basic Case for 10.or E (...,1,...,multicolor,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [40]:
xyz1=xyz[['group','deprecated_hack_merchant_id']].drop_duplicates()
xyz.drop(columns='deprecated_hack_merchant_id',drop=True)
xyz=xyz.merge(xyz,xyz1,on='group',how='left')

Index(['asin', 'parent_asin', 'ptd', 'brand_name', 'attribute_list',
       'attribute_value_list', 'attribute_list_var',
       'attribute_value_list_var', 'item_name', 'comp_flag',
       'orphan_variation', 'split_asin_count', 'gvs', 'run_dt',
       'gl_product_group_desc', 'recommendation', 'parent_rep', 'group',
       'count_parent', 'recommended_parent_asin', 'Status', 'TRP',
       'Variation_theme', 'Title_Main_output', 'Variation_theme_Main_output',
       'color.value_x', 'material.value_x', 'size.value_x',
       'deprecated_hack_merchant_id', 'asin2', 'color#1.value_x',
       'color.value_y', 'frame#1.type#1.value_x', 'material#1.value_x',
       'material.value_y', 'material_type_free#1.value_x',
       'occasion_type#1.value_x', 'size#1.value_x', 'size.value_y',
       'team_name#1.value_x', 'team_name.value_x', 'color#1.value_y',
       'color.value', 'frame#1.type#1.value_y', 'material#1.value_y',
       'material.value', 'material_type_free#1.value_y',
       'occas

In [38]:
#df1=pd.read_csv('Solimo - Aug 26 - workfile_RootTool.csv',encoding='latin')

##df1=df[df['Status']=='Create New Parent']
df1['asin2']=df1['asin']

result = []
for name, group_df in df1.groupby('recommended_parent_asin', sort=False):
    label_row = {col: '' for col in df1.columns}
    label_row['asin']=name
    label_row['ptd']=group_df['ptd'].drop_duplicates()[group_df['ptd'].drop_duplicates().index[0]]
    label_row['brand_name']=group_df['brand_name'].drop_duplicates()[group_df['brand_name'].drop_duplicates().index[0]]
    label_row['item_name']=group_df['Title_Main_output'].drop_duplicates()[group_df['Title_Main_output'].drop_duplicates().index[0]]
    result.append(label_row)
    result.extend(group_df.to_dict(orient='records'))
xyz=pd.DataFrame(result)
xyz
xyz['Variation_theme_Main_output'].replace(',','/',inplace=True,regex=True)
xyz_3p=xyz[xyz['deprecated_hack_merchant_id'].isna()]
blank_temp=pd.read_excel("Plantex+-+create+new+families+2(1).xlsm",sheet_name='Template',nrows=5,header=None)
header=blank_temp.iloc[4]
blank_temp=blank_temp.rename(columns=header)
#blank_temp.loc[0,:]=blank_temp.columns
blank_temp=blank_temp.iloc[5:]

blank_temp['contribution_sku#1.value']=xyz_3p['asin']
blank_temp['product_type#1.value']=xyz_3p['ptd']
blank_temp['item_name[marketplace_id=A21TJRUUN4KGV][language_tag=en_IN]#1.value']=xyz_3p['item_name']
blank_temp['brand[marketplace_id=A21TJRUUN4KGV][language_tag=en_IN]#1.value']=xyz_3p['brand_name']
blank_temp['amzn1.volt.ca.product_id_value']=xyz_3p['asin2']
blank_temp['::record_action']=np.where(blank_temp['amzn1.volt.ca.product_id_value']=='','Create or Replace (Full Update)','Edit (Partial Update)')
blank_temp['parentage_level[marketplace_id=A21TJRUUN4KGV]#1.value']=np.where(blank_temp['amzn1.volt.ca.product_id_value']=='','Parent','Child')
blank_temp['child_parent_sku_relationship[marketplace_id=A21TJRUUN4KGV]#1.child_relationship_type']=np.where(blank_temp['amzn1.volt.ca.product_id_value']=='','','Variation')
blank_temp['child_parent_sku_relationship[marketplace_id=A21TJRUUN4KGV]#1.parent_sku']=xyz_3p['asin2']
blank_temp['country_of_origin[marketplace_id=A21TJRUUN4KGV]#1.value']=np.where(blank_temp['amzn1.volt.ca.product_id_value']=='','India','')
blank_temp['batteries_required[marketplace_id=A21TJRUUN4KGV]#1.value']=np.where(blank_temp['amzn1.volt.ca.product_id_value']=='','No','')
blank_temp['variation_theme#1.name']=xyz_3p['Variation_theme_Main_output']
blank_temp['variation_theme#1.name'].fillna(method='bfill',inplace=True)
blank_temp

/tmp/ipykernel_20945/3432781826.py:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  xyz['Variation_theme_Main_output'].replace(',','/',inplace=True,regex=True)
/tmp/ipykernel_20945/3432781826.py:37: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].met

,contribution_sku#1.value,::record_action,product_type#1.value,item_name[marketplace_id=A21TJRUUN4KGV][language_tag=en_IN]#1.value,brand[marketplace_id=A21TJRUUN4KGV][language_tag=en_IN]#1.value,amzn1.volt.ca.product_id_type,amzn1.volt.ca.product_id_value,recommended_browse_nodes[marketplace_id=A21TJRUUN4KGV]#1.value,recommended_browse_nodes[marketplace_id=A21TJRUUN4KGV]#2.value,recommended_browse_nodes[marketplace_id=A21TJRUUN4KGV]#3.value,...,item_display_weight[marketplace_id=A21TJRUUN4KGV]#1.value,item_display_weight[marketplace_id=A21TJRUUN4KGV]#1.unit,master_pack_layers_per_pallet_quantity[marketplace_id=A21TJRUUN4KGV]#1.value,master_packs_per_layer_quantity[marketplace_id=A21TJRUUN4KGV]#1.value,item_dimensions[marketplace_id=A21TJRUUN4KGV]#1.length.value,item_dimensions[marketplace_id=A21TJRUUN4KGV]#1.length.unit,item_dimensions[marketplace_id=A21TJRUUN4KGV]#1.width.value,item_dimensions[marketplace_id=A21TJRUUN4KGV]#1.width.unit,item_dimensions[marketplace_id=A21TJRUUN4KGV]#1.height.value,item_dimensions[marketplace_id=A21TJRUUN4KGV]#1.height.unit
983,B0DTJ542DD,Edit (Partial Update),CELLULAR_PHONE_CASE,Amazon Brand - Solimo Ultra Hybrid Clear Camer...,Amazon Brand - Solimo,NaN,B0DTJ542DD,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
984,B0DTJ5MJYY,Edit (Partial Update),CELLULAR_PHONE_CASE,Amazon Brand - Solimo Ultra Hybrid Clear Camer...,Amazon Brand - Solimo,NaN,B0DTJ5MJYY,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3101,B0DSC9MRT2,Edit (Partial Update),BACKPACK,Amazon Brand - Solimo School Bag for Kids Boys...,Amazon Brand - Solimo,NaN,B0DSC9MRT2,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3102,B0DSCB3BBR,Edit (Partial Update),BACKPACK,Amazon Brand - Solimo School Bag for Kids Boys...,Amazon Brand - Solimo,NaN,B0DSCB3BBR,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3103,B0DSC8JWYZ,Edit (Partial Update),BACKPACK,Amazon Brand - Solimo School Bag for Kids Boys...,Amazon Brand - Solimo,NaN,B0DSC8JWYZ,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3104,B0DSC91V61,Edit (Partial Update),BACKPACK,Amazon Brand - Solimo School Bag for Kids Boys...,Amazon Brand - Solimo,NaN,B0DSC91V61,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3105,B0DSC7LV6M,Edit (Partial Update),BACKPACK,Amazon Brand - Solimo School Bag for Kids Boys...,Amazon Brand - Solimo,NaN,B0DSC7LV6M,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3106,B0DSC7627Y,Edit (Partial Update),BACKPACK,Amazon Brand - Solimo School Bag for Kids Boys...,Amazon Brand - Solimo,NaN,B0DSC7627Y,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3107,B0DSCBVWHC,Edit (Partial Update),BACKPACK,Amazon Brand - Solimo School Bag for Kids Boys...,Amazon Brand - Solimo,NaN,B0DSCBVWHC,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3108,B0DSC8FLM3,Edit (Partial Update),BACKPACK,Amazon Brand - Solimo School Bag for Kids Boys...,Amazon Brand - Solimo,NaN,B0DSC8FLM3,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [26]:
## Create new template
df1=pd.read_csv('Solimo - Aug 26 - workfile_RootTool.csv',encoding='latin')

##df1=df[df['Status']=='Create New Parent']
df1['asin2']=df1['asin']

result = []
for name, group_df in df1.groupby('recommended_parent_asin', sort=False):
    label_row = {col: '' for col in df1.columns}
    label_row['asin']=name
    label_row['ptd']=group_df['ptd'].drop_duplicates()[group_df['ptd'].drop_duplicates().index[0]]
    label_row['brand_name']=group_df['brand_name'].drop_duplicates()[group_df['brand_name'].drop_duplicates().index[0]]
    label_row['item_name']=group_df['Title_Main_output'].drop_duplicates()[group_df['Title_Main_output'].drop_duplicates().index[0]]
    label_row['group']=group_df['group'].drop_duplicates()[group_df['group'].drop_duplicates().index[0]]
    result.append(label_row)
    result.extend(group_df.to_dict(orient='records'))
xyz=pd.DataFrame(result)
xyz1=xyz[['group','deprecated_hack_merchant_id']].drop_duplicates()
xyz.drop(columns='deprecated_hack_merchant_id',drop=True)
xyz=xyz.merge(xyz,xyz1,on='group',how='left')
xyz['Variation_theme_Main_output'].replace(',','/',inplace=True,regex=True)
xyz_3p=xyz[xyz['deprecated_hack_merchant_id'].isna()]
blank_temp=pd.read_excel("Plantex+-+create+new+families+2(1).xlsm",sheet_name='Template',nrows=5,header=None)
header=blank_temp.iloc[4]
blank_temp=blank_temp.rename(columns=header)
#blank_temp.loc[0,:]=blank_temp.columns
blank_temp=blank_temp.iloc[5:]

blank_temp['contribution_sku#1.value']=xyz_3p['asin']
blank_temp['product_type#1.value']=xyz_3p['ptd']
blank_temp['item_name[marketplace_id=A21TJRUUN4KGV][language_tag=en_IN]#1.value']=xyz_3p['item_name']
blank_temp['brand[marketplace_id=A21TJRUUN4KGV][language_tag=en_IN]#1.value']=xyz_3p['brand_name']
blank_temp['amzn1.volt.ca.product_id_value']=xyz_3p['asin2']
blank_temp['::record_action']=np.where(blank_temp['amzn1.volt.ca.product_id_value']=='','Create or Replace (Full Update)','Edit (Partial Update)')
blank_temp['parentage_level[marketplace_id=A21TJRUUN4KGV]#1.value']=np.where(blank_temp['amzn1.volt.ca.product_id_value']=='','Parent','Child')
blank_temp['child_parent_sku_relationship[marketplace_id=A21TJRUUN4KGV]#1.child_relationship_type']=np.where(blank_temp['amzn1.volt.ca.product_id_value']=='','','Variation')
blank_temp['child_parent_sku_relationship[marketplace_id=A21TJRUUN4KGV]#1.parent_sku']=xyz_3p['asin2']
blank_temp['country_of_origin[marketplace_id=A21TJRUUN4KGV]#1.value']=np.where(blank_temp['amzn1.volt.ca.product_id_value']=='','India','')
blank_temp['batteries_required[marketplace_id=A21TJRUUN4KGV]#1.value']=np.where(blank_temp['amzn1.volt.ca.product_id_value']=='','No','')
blank_temp['variation_theme#1.name']=xyz_3p['Variation_theme_Main_output']
blank_temp['variation_theme#1.name'].fillna(method='bfill',inplace=True)
blank_temp
blank_temp1=pd.read_excel("Plantex+-+create+new+families+2(1).xlsm",sheet_name='Template',nrows=4,header=None)
blank_temp1
new_df=pd.concat([blank_temp1.transpose(),blank_temp.transpose().reset_index()],axis=1,ignore_index=True).transpose()
# for i in xyz.columns(3:):
#     k=np.where(new_df.iloc[1,:]==k)[0][0]
#     new_df.iloc[:,k]=list(pd.concat([dff.iloc[:,k],df4_3P['Asin']]))
new_df.columns=new_df.iloc[0,:]
new_df=new_df.iloc[1:]
new_df.to_csv("variation_3p_template.csv",index=False)

#Ra
xyz=xyz[xyz['deprecated_hack_merchant_id'].notna()]
blank_temp_ret=pd.read_excel("Sample pre-filled template for Create new families.xlsx",sheet_name='Sheet0')
blank_temp_ret.columns
blank_temp_ret1=blank_temp_ret.copy()
blank_temp_ret1.columns=blank_temp_ret1.iloc[0,:]
blank_temp_ret1=blank_temp_ret1.drop(index=0)
blank_temp_ret1['asin']=xyz['asin']
blank_temp_ret1['sc_vendor_name']='AmazonIn/PDHEU'
blank_temp_ret1['brand.value']=xyz['brand_name']
blank_temp_ret1['gl_product_group_type.value']=xyz['gl_product_group_desc']
blank_temp_ret1['item_name.value']=xyz['item_name']
blank_temp_ret1['product_type.value']=xyz['ptd']
blank_temp_ret1['rtip_product_line.value']=xyz['ptd']
blank_temp_ret1['$row_type']=blank_temp['parentage_level[marketplace_id=A21TJRUUN4KGV]#1.value'].replace(['Parent','Child'],['P','C'])
blank_temp_ret1=pd.concat([blank_temp_ret1,xyz[var7.columns.drop('asin')]],axis=1)
blank_temp_ret1['deprecated_variation_theme.name']=xyz['Variation_theme_Main_output']
blank_temp_ret1['deprecated_variation_theme.name']=np.where(blank_temp_ret1['deprecated_variation_theme.name']=='',np.nan,blank_temp_ret1['deprecated_variation_theme.name'])
blank_temp_ret1['deprecated_variation_theme.name'].fillna(method='bfill',inplace=True)

blank_temp_ret.drop(index=0,inplace=True)
blank_temp_ret=pd.concat([blank_temp_ret.transpose().reset_index(),blank_temp_ret1.transpose().reset_index(names='new_index')],axis=1,ignore_index=True).transpose()
blank_temp_ret.columns=blank_temp_ret.iloc[0,:]
blank_temp_ret.drop(index=0,inplace=True)
blank_temp_ret.columns=['' if (str(i).startswith('Unnamed')) | (i==np.nan) else i for i in blank_temp_ret.columns]

blank_temp_ret.to_csv("Retail_Template.csv",index=False)

/tmp/ipykernel_26627/98241869.py:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  xyz['Variation_theme_Main_output'].replace(',','/',inplace=True,regex=True)
/tmp/ipykernel_26627/98241869.py:37: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(

In [26]:

# blank_temp_ret.to_csv("xxx.csv",index=False)

df1=pd.read_csv('Solimo - Aug 26 - workfile_RootTool.csv',encoding='latin')
df1

,asin,parent_asin,ptd,brand_name,attribute_list,attribute_value_list,attribute_list_var,attribute_value_list_var,item_name,comp_flag,...,recommended_parent_asin,Status,TRP,Variation_theme,Title_Main_output,Variation_theme_Main_output,color.value,material.value,size.value,deprecated_hack_merchant_id
0,B098B51D9N,NaN,CELLULAR_PHONE_CASE,Amazon Brand - Solimo,brand_name|compatible_phone_models|form_factor...,amazon brand - solimo|10.or e|basic case|silic...,color_name_varattr,multicolor,Amazon Brand - Solimo Basic Case for 10.or E (...,1,...,Create New parent,Create New parent,brand_name+compatible_phone_models+form_factor...,color_name,Amazon Brand - Solimo Basic Case for 10.or E,color_name,NaN,NaN,NaN,8.302773e+08
1,B098B7QD2S,NaN,CELLULAR_PHONE_CASE,Amazon Brand - Solimo,brand_name|compatible_phone_models|form_factor...,amazon brand - solimo|10.or e|basic case|silic...,color_name_varattr,multicolor,Amazon Brand - Solimo Basic Case for 10.or E (...,1,...,Create New parent,Create New parent,brand_name+compatible_phone_models+form_factor...,color_name,Amazon Brand - Solimo Basic Case for 10.or E,color_name,NaN,NaN,NaN,8.302773e+08
2,B08W58Y7YG,NaN,CELLULAR_PHONE_CASE,Amazon Brand - Solimo,brand_name|compatible_phone_models|form_factor...,amazon brand - solimo|2020|basic case|plastic|...,color_name_varattr,multicolor,Amazon Brand - Solimo Designer Game Remote 3D ...,1,...,Create New parent,Create New parent,brand_name+compatible_phone_models+form_factor...,color_name,Amazon Brand - Solimo Designer 3D Printed Hard...,color_name,NaN,NaN,NaN,8.302773e+08
3,B08SKDZ4B1,NaN,CELLULAR_PHONE_CASE,Amazon Brand - Solimo,brand_name|compatible_phone_models|form_factor...,amazon brand - solimo|2020|basic case|plastic|...,color_name_varattr,multicolor,Amazon Brand - Solimo Designer Golden Stars UV...,1,...,Create New parent,Create New parent,brand_name+compatible_phone_models+form_factor...,color_name,Amazon Brand - Solimo Designer 3D Printed Hard...,color_name,NaN,NaN,NaN,8.302773e+08
4,B08W58HGSB,NaN,CELLULAR_PHONE_CASE,Amazon Brand - Solimo,brand_name|compatible_phone_models|form_factor...,amazon brand - solimo|2020|basic case|plastic|...,color_name_varattr,multicolor,Amazon Brand - Solimo Designer Peacock Feather...,1,...,Create New parent,Create New parent,brand_name+compatible_phone_models+form_factor...,color_name,Amazon Brand - Solimo Designer 3D Printed Hard...,color_name,NaN,NaN,NaN,8.302773e+08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7218,B0B34Q2HT3,NaN,CELLULAR_PHONE_CASE,Amazon Brand - Solimo,brand_name|compatible_phone_models|form_factor...,amazon brand - solimo|xiomi 12 pro 5g|basic ca...,color_name_varattr,multicolor,Amazon Brand- Solimo Basic Case for Xiomi 12 P...,1,...,Create New parent,Create New parent,brand_name+compatible_phone_models+form_factor...,color_name,Amazon Brand- Solimo Basic Case for Xiomi 12 P...,color_name,NaN,NaN,NaN,1.104436e+09
7219,B0D1VFHQV9,B0DHRQ9WD8,STORAGE_BAG,Amazon Brand - Solimo,brand_name|closure_type|item_shape|material_ty...,amazon brand - solimo|zipper|rectangular|cotto...,color_name|size_name_varattr,cream|pack of 20,Amazon Brand - Solimo Saree Bags | Clothes Bag...,1,...,B0DHRQ9WD8,To be Retained,brand_name+closure_type+item_shape+material_ty...,color_name+size_name,Amazon Brand - Solimo Hanging Saree Bag | Cott...,"color_name,size_name",cream,NaN,NaN,7.405493e+10
7220,B0D1VFXNV8,NaN,STORAGE_BAG,Amazon Brand - Solimo,brand_name|closure_type|item_shape|material_ty...,amazon brand - solimo|zipper|rectangular|cotto...,color_name|size_name_varattr,cream|pack of 15,Amazon Brand - Solimo Hanging Saree Bag | Cott...,1,...,B0DHRQ9WD8,To be tagged,brand_name+closure_type+item_shape+material_ty...,color_name+size_name,Amazon Brand - Solimo Hanging Saree Bag | Cott...,"color_name,size_name",cream,NaN,NaN,7.405493e+10
7221,B0BV9F8F53,NaN,BOTTLE,Amazon Brand - Solimo,bottle_type|brand_name|material_type,wide-neck|amazon brand - solimo|polyethylene t...,colo

In [27]:
pd.concat([blank_temp_ret1,xyz[var7.columns.drop('asin')]],axis=1)
#blank_temp_ret1
#xyz[var7.columns.drop('asin')]

,$row_type,asin,sc_vendor_name,brand.value,deprecated_variation_theme.name,item_name.value,product_type.value,rtip_product_line.value,gl_product_group_type.value,color.value,material.value,size.value,color.value,material.value,size.value
0,NaN,Create New parent,AmazonIn/PDHEU,Amazon Brand - Solimo,color_name,Amazon Brand - Solimo Basic Case for 10.or E,CELLULAR_PHONE_CASE,CELLULAR_PHONE_CASE,,,,,,,
1,NaN,B098B51D9N,AmazonIn/PDHEU,Amazon Brand - Solimo,color_name,Amazon Brand - Solimo Basic Case for 10.or E (...,CELLULAR_PHONE_CASE,CELLULAR_PHONE_CASE,gl_wireless_accessory,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,B098B7QD2S,AmazonIn/PDHEU,Amazon Brand - Solimo,color_name,Amazon Brand - Solimo Basic Case for 10.or E (...,CELLULAR_PHONE_CASE,CELLULAR_PHONE_CASE,gl_wireless_accessory,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,B08W58Y7YG,AmazonIn/PDHEU,Amazon Brand - Solimo,color_name,Amazon Brand - Solimo Designer Game Remote 3D ...,CELLULAR_PHONE_CASE,CELLULAR_PHONE_CASE,gl_wireless_accessory,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,B08SKDZ4B1,AmazonIn/PDHEU,Amazon Brand - Solimo,color_name,Amazon Brand - Solimo Designer Golden Stars UV...,CELLULAR_PHONE_CASE,CELLULAR_PHONE_CASE,gl_wireless_accessory,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7235,NaN,B084QRP5R6,AmazonIn/PDHEU,Amazon Brand - Solimo,color_name,Amazon Brand - Solimo Designer Love Pink UV Pr...,CELLULAR_PHONE_CASE,CELLULAR_PHONE_CASE,gl_wireless_accessory,NaN,NaN,NaN,NaN,NaN,NaN
7236,NaN,B084QR5P68,AmazonIn/PDHEU,Amazon Brand - Solimo,color_name,Amazon Brand - Solimo Designer Stone Heart UV ...,CELLULAR_PHONE_CASE,CELLULAR_PHONE_CASE,gl_wireless_accessory,NaN,NaN,NaN,NaN,NaN,NaN
7237,NaN,B0DHRQ9WD8,AmazonIn/PDHEU,Amazon Brand - Solimo,color_name/size_name,Amazon Brand - Solimo Hanging Saree Bag | Cott...,STORAGE_BAG,STORAGE_BAG,,,,,,,
7238,NaN,B0D1VFHQV9,AmazonIn/PDHEU,Amazon Brand - Solimo,color_name/size_name,Amazon Brand - Solimo Saree Bags | Clothes Bag...,STORAGE_BAG,STORAGE_BAG,gl_home,cream,NaN,NaN,cream,NaN,NaN


In [18]:
df2[df2['Set']=='set 101']['Main_output'][3]
df2[df2['Main_output'].str.contains('<')]

,Set,TRP,Title,prompt,output,Main_output


In [39]:

#### Titles

import pandas as pd
import numpy as np
import tqdm
from typing import *

import numpy as np
import pandas as pd
from display import *
from call_bedrock_fast_claude_v3_py import *
import sagemaker
import boto3
from sagemaker import get_execution_role

system='''Consider yourself as a Catalog Category expert.Your task is to read the list of titles in <inp></inp>\
and create a generalize title as per the TRP in <att></att>.Remove all other values from title which are not matching the TRP in <att><att>.\
.Each title is separated by "<>" in\
<inp></inp> represents individual title and attribute is separated by "+" . Pick titles in <inp></inp> and use your intelligence and own\
valid value repository to identify the attribute which needs to be retained\
from list of attributes in <att></att>.\
use your judgement similar to the following the <thinking></thinking> in last <example></example>. 
Respond the new title in <res></res>.\
If you dont find any title then respond NA.\
If you dont find any attribute value in any of the titles then respond "value_missing(attr)".If you find multiple vlaid values for same attribute in different title then resepond NA
Make sure to output your answer for one input in <res></res> only.Please make sure to not output in format of <res>val</res>,<res>val2</res>.You can refer <example></example> for reference input out put instruction.\

<example>
<att>item_type_name + compatible_phone_models + material_type</att>
<inp>Casotec Rangoli Paisley Art Design Printed Silicon Soft TPU Back Case Cover for 10.or D2</inp>
<res>Casotec Printed Silicon Soft TPU Back Case Cover for 10.or D2</res>
</example>

<example>
<att>item_type_name + compatible_phone_models + material_type</att>
<inp>Casotec Basic Case for 10.or D2 4G (Silicone_Multicolor)</inp>
<res> Casotec Basic Case for 10.or D2 4G</res>
</example>

<example>
<att>item_type_name + compatible_phone_models + material_type</att>
<inp>Casotec Bad Color Shape Design Printed Silicon Soft TPU Back Case Cover for Xolo Era<>
Casotec Basic Case for Xolo Era (Silicone_Multicolor)<>
Casotec Basic Case for Xolo Era (Silicone_Multicolor)
</inp>
<thinking>From the input given as per TRP I can find Back Case as most relevant for item_type_name, Xolo Era for compatible_devices
,Silicon Soft TPU Silicon Soft TPU  for material. Removing all extra data and creating best general title will be "Casotec Printed Silicon Soft TPU Back Case Cover for Xolo Era"</thinking>
<res>Casotec Printed Silicon Soft TPU Back Case Cover for Xolo Era</res>
</example>

<example>
<att>brand_name+material_type+model_name</att>
<inp>Bosch Hand Blender MS1BG1020I 400 W with Beaker and Chopper (Black)<>Bosch Hand Blender MS1WR0000I 300 W (White)</inp>
<thinking> From the input given as per TRP I can find brand as Bosch, no relevant values for material_type, no values for model_name. As instructed since attribute value is missing in title the correct output will be "value_missing(material_type,model_name)"
<res>value_missing(material_type,model_name)</res>
</example>

<example>
'<att>brand_name+material_type</att>\n<inp>Amazon Brand - Umi Brass Hanuman Idol, 8 X 5 X 4 Inches<>Amazon Brand - Umi Romantic Love Couple Showpiece with Lighting Gift for Love Couple Wife Girlfriend, Teddy Bear Showpiece, Anniversary Wedding Valentine Day Gift for Him Her (Pink)<>Amazon Brand - Umi Polyresin Pagdi Ganesha Murti Idol Statue for Car Dashboard, Ganesha Chaturthi Decoration Gift Item<>Amazon Brand - Umi Ganesh Idol for Car Dashboard, Ganesha Ganpati Idol for Home Gift, Terracotta Statue Murti for Car Home Decor Temple Puja Room Gift</inp>'
<thinking> From the input given as per TRP I can find brand as Amazon Brand - Umi, there are different relevant values in different title. As instructed since there are more then 1 relevane value for material type I recommend title as "NA"
<res>NA</res>
</example>

'''.strip()


prompt2='''<att>{a}</att>
<inp>{inp}</inp>
'''.strip()



df1=pd.read_excel('Solimo - Oct - parent title generation.xlsx')
#sets=['Set'+str(i) for i in range(4227,len(df1))]
#print(sets)
#df1=df1[df1['Set'].isin(sets)]
#df1=df1[df1['PT']=='ELECTRONIC_DEVICE_SKIN']

df4=df1[df1['Set']=='Set1']
inp="<>".join(list(df4['Title']))
a=df1['TRP'].unique()
prompt=system.format(a=a,inp=inp)

#system
df2=df1.groupby(['Set','TRP'],as_index=False)['Title'].apply(lambda x:'<>'.join(x))
df2['TRP'].replace('\|','+',regex=True,inplace=True)

df2['prompt']=df2.apply(lambda x:prompt2.format(a=x['TRP'],inp=x['Title']),axis=1).astype('str')
#df1['prompt']=df1.apply(lambda x:)
#df2


ad=pd.Series()

df2['output']=call_bedrock_fast(
            prompts=df2['prompt'],  ## REPLACE THIS WITH YOUR PROMPT!!
            model="anthropic.claude-3-haiku-20240307-v1:0",
            system=system,  ## Only works with Claude v3
            max_new_tokens=200,
            temperature=0.9,
        
        #top_p=1.0

            
        )

from bs4 import BeautifulSoup as soup
#b=soup.find_all('<res>')
def ext(x,tag_name='p'):
    r=soup(str(x))
    c=r.find_all(tag_name)
    return str(c)
df2['Main_output']=df2['output'].apply(ext,tag_name='res')
df2['Main_output'].replace(['\[<res>','</res>\]'],"",regex=True,inplace=True)
df2.to_csv('Title_Solimo.csv',index=False)
df2

/tmp/ipykernel_29361/389608333.py:86: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df2['TRP'].replace('\|','+',regex=True,inplace=True)
100%|███████████████████████████████████████████████████████████| 7876/7876 [11:06<00:00, 11.81it/s]
/tmp/ipykernel_29361/389608333.py:114: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are s

,Set,TRP,Title,prompt,output,Main_output
0,set 1,brand_name+material_type+model_name,Amazon Brand - Solimo School Bag for Kids Boys...,<att>brand_name+material_type+model_name</att>...,<res>Amazon Brand - Solimo School Bag for Kids...,Amazon Brand - Solimo School Bag for Kids Boys...
1,set 10,brand_name+item_shape+material_type+recommende...,Amazon Brand - Solimo Premium Faux Suede Butte...,<att>brand_name+item_shape+material_type+recom...,<res>Amazon Brand - Solimo Premium Faux Suede ...,Amazon Brand - Solimo Premium Faux Suede Butte...
2,set 100,brand_name+compatible_phone_models+form_factor...,Amazon Brand - Solimo Designer Golden Sparkle ...,<att>brand_name+compatible_phone_models+form_f...,<res>Amazon Brand - Solimo Designer UV Printed...,Amazon Brand - Solimo Designer UV Printed Soft...
3,set 1000,brand_name+compatible_phone_models+form_factor...,Amazon Brand- Solimo Basic Case for Apple iPho...,<att>brand_name+compatible_phone_models+form_f...,<res>Amazon Brand- Solimo Basic Case for Apple...,Amazon Brand- Solimo Basic Case for Apple iPho...
4,set 1001,brand_name+compatible_phone_models+form_factor...,Amazon Brand- Solimo Basic Case for Apple iPho...,<att>brand_name+compatible_phone_models+form_f...,<res>Amazon Brand- Solimo Basic Case for Apple...,Amazon Brand- Solimo Basic Case for Apple iPho...
...,...,...,...,...,...,...
7871,set 995,brand_name+compatible_phone_models+form_factor...,Amazon Brand- Solimo Basic Case for Apple iPho...,<att>brand_name+compatible_phone_models+form_f...,<res>Amazon Brand- Solimo Basic Case for Apple...,Amazon Brand- Solimo Basic Case for Apple iPho...
7872,set 996,brand_name+compatible_phone_models+form_factor...,Amazon Brand - Solimo Designer Series UV Print...,<att>brand_name+compatible_phone_models+form_f...,<res>Amazon Brand - Solimo Designer Series UV ...,Amazon Brand - Solimo Designer Series UV Print...
7873,set 997,brand_name+compatible_phone_models+form_factor...,Amazon Brand - Solimo Designer Two Number 3D P...,<att>brand_name+compatible_phone_models+form_f...,<res>Amazon Brand - Solimo 3D Printed Hard Bac...,Amazon Brand - Solimo 3D Printed Hard Back Cas...
7874,set 998,brand_name+compatible_phone_models+form_factor...,Amazon Brand - Solimo Basic Case for Apple iPh...,<att>brand_name+compatible_phone_models+form_f...,<res>Amazon Brand - Solimo Basic Case for Appl...,Amazon Brand - Solimo Basic Case for Apple iPh...


In [ ]:
df2[]

In [ ]:
system='''Consider yourself as a Catalog Category expert.Your task is to read the list of titles in <inp></inp> identify attribute which can be used to create variations from given list of attribute provided in <att></att>\
.Each title is separated by "<>" in \
<inp></inp> represents individual title and attribute is separated by "/" . Pick titles in <inp></inp> and use your intelligence and own valid value repository to identify the attribute using which they can be varieted  \
from list of attributes in <att></att>.\
Once you find a attribute\
for an input you can respond in output <res></res>.Incase you get more than one attribute for the input then respond in comma separated format.If you get valid values for an attribute and it is same for all asins in 1 group then do consider that attribute for variation theme.\
If you dont find any attribte then respond NA.\
Make sure to output your answer for one input in <res></res>.Stop responding after output of attribute.\
Please do not give any output apart from list provided in <att></att>. 


<att>{a}</att>
<inp>{inp}</inp>
'''.strip()

prompt2='''<att>{a}</att>
<inp>{inp}</inp>
'''.strip()
df=pd.read_csv('sample_var_output.csv')
df1=df[(df['pa2'].notna()) & (df['pa2']!='Make Standalone')]
df1
var_th=pd.read_csv('variation_theme.csv')
var_th.rename(columns={'PTD','ptd'})
df1=pd.merge(df1,var_th,on='ptd',how='left')

In [ ]:
df2=df1.groupby(['group','Variation_theme'],as_index=False)['Title'].apply(lambda x:'<>'.join(x))
df2['TRP'].replace('\|','+',regex=True,inplace=True)

df2['prompt']=df2.apply(lambda x:prompt2.format(a=x['TRP'],inp=x['Title']),axis=1).astype('str')
ad=pd.Series()

df2['output']=call_bedrock_fast(
            prompts=df2['prompt'],  ## REPLACE THIS WITH YOUR PROMPT!!
            model="anthropic.claude-3-haiku-20240307-v1:0",
            system=system,  ## Only works with Claude v3
            max_new_tokens=200,
            temperature=0.9,
        
        #top_p=1.0

            
        )

from bs4 import BeautifulSoup as soup
#b=soup.find_all('<res>')
def ext(x,tag_name='p'):
    r=soup(str(x))
    c=r.find_all(tag_name)
    return str(c)
df2['Main_output']=df2['output'].apply(ext,tag_name='res')


for i in df1['group'].drop_duplicates():
    df4=df1[df1['group']==i]
    inp="<>".join(list(df4['item_name']))
    a=var_th[var_th['PTD']==df4['ptd'].unique()[0]]['Variation_theme']
    prompt=sytem.format(a=a,inp=inp)
    import boto3
    import json

    bedrock = boto3.client(service_name="bedrock-runtime")
    body = json.dumps({
      "max_tokens": 256,
      "messages": [{"role": "user", "content": prompt}],
      "anthropic_version": "bedrock-2023-05-31",
      'temperature':0.1,
        'top_k':8 
    })

    response = bedrock.invoke_model(body=body, modelId="anthropic.claude-3-haiku-20240307-v1:0")

    response_body = json.loads(response.get("body").read())
    print(response_body.get("content"),i,df4['ptd'].unique()[0])